In [239]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [240]:
df=pd.read_csv('../train_prepared.csv')
df.head()

,region,parent_category_name,category_name,param_1,param_2,param_3,price,item_seq_number,user_type,image,deal_probability,has_params,description_len,description_group
0,Свердловская область,Личные вещи,Товары для детей и игрушки,1,0,0,400.00,2,Private,1,0.13,1,58,3. Оптимальное
1,Самарская область,Для дома и дачи,Мебель и интерьер,1,0,0,3000.00,19,Private,1,0.00,1,41,2. Очень короткое
2,Ростовская область,Бытовая электроника,Аудио и видео,1,0,0,4000.00,9,Private,1,0.43,1,99,3. Оптимальное
3,Татарстан,Личные вещи,Товары для детей и игрушки,1,0,0,2200.00,286,Company,1,0.80,1,22,2. Очень короткое
4,Волгоградская область,Транспорт,Автомобили,1,1,1,40000.00,3,Private,1,0.21,1,24,2. Очень короткое


In [241]:
df.groupby('category_name')['category_name'].count().describe()

count     47.00
mean    1046.11
std     1852.39
min        2.00
25%      229.00
50%      362.00
75%      940.50
max     9232.00
Name: category_name, dtype: float64

In [242]:
liq_low_ad=df.groupby('category_name')[['category_name','deal_probability']].agg(
    num_of_ad=('category_name', 'count'),
    avg_probability=('deal_probability', 'mean')
).reset_index()
liq_low_ad=liq_low_ad.query("num_of_ad<362")
liq_low_ad=liq_low_ad.sort_values(by='avg_probability', ascending=False)
liq_low_ad

,category_name,num_of_ad,avg_probability
23,Мотоциклы и мототехника,202,0.25
5,Велосипеды,321,0.24
36,Птицы,197,0.23
6,Водный транспорт,80,0.21
27,Ноутбуки,332,0.20
24,Музыкальные инструменты,220,0.19
9,Грузовики и спецтехника,324,0.19
30,Оргтехника и расходники,221,0.18
25,Настольные компьютеры,157,0.18
32,Планшеты и электронные книги,284,0.17


In [243]:
for param in ['param_1', 'param_2', 'param_3']:
    grouped=df.groupby(param)['deal_probability'].mean().reset_index()

    diff=grouped.max()-grouped.min()

    print(f"Difference for {grouped}:{diff}")

Difference for    param_1  deal_probability
0        0              0.15
1        1              0.14:param_1            1.00
deal_probability   0.02
dtype: float64
Difference for    param_2  deal_probability
0        0              0.17
1        1              0.12:param_2            1.00
deal_probability   0.05
dtype: float64
Difference for    param_3  deal_probability
0        0              0.17
1        1              0.09:param_3            1.00
deal_probability   0.08
dtype: float64


In [244]:
matrix_cat_des=df.groupby(['category_name','description_group'])['deal_probability'].mean().unstack()

best_group=matrix_cat_des.idxmax(axis=1)
max_val=matrix_cat_des.max(axis=1)

drop=max_val-matrix_cat_des['5. Слишком длинное']

report = pd.DataFrame({
    'Золотая середина': best_group,
    'Макс. ликвидность': max_val,
    'Падение на длинных текстах': drop
}).reset_index()

report

,category_name,Золотая середина,Макс. ликвидность,Падение на длинных текстах
0,Автомобили,4. Подробное,0.31,0.20
1,Аквариум,3. Оптимальное,0.17,0.07
2,Аудио и видео,2. Очень короткое,0.20,0.08
3,Билеты и путешествия,3. Оптимальное,0.25,0.20
4,Бытовая техника,3. Оптимальное,0.30,0.26
5,Велосипеды,1. Пусто,0.62,0.58
6,Водный транспорт,2. Очень короткое,0.23,0.23
7,Гаражи и машиноместа,1. Пусто,0.25,NaN
8,Готовый бизнес,2. Очень короткое,0.49,0.46
9,Грузовики и спецтехника,2. Очень короткое,0.20,0.14


In [245]:
df['median_price']=df.groupby('category_name')['price'].transform('median')
df['price_ratio']=df['price']/df['median_price']

bins = [0, 0.5, 0.8, 1.1, 1.5, 2, np.inf]
labels = [
    '1. Намного дешевле рынка (меньше половины цены)',
    '2. Скидка (на 20-50% дешевле медианы)',
    '3. Рыночная цена (около медианы +/- 10%)',
    '4. Чуть дороже рынка (на 10-50% дороже)',
    '5. Намного дороже рынка (в 1.5 - 2 раза дороже)',
    '6. Оверпрайс (более чем в 2 раза дороже)'
]

df['price_segment']=pd.cut(df['price_ratio'], bins=bins, labels=labels)
price_impact=df.groupby('price_segment')['deal_probability'].mean()
price_impact

price_segment
1. Намного дешевле рынка (меньше половины цены)   0.15
2. Скидка (на 20-50% дешевле медианы)             0.13
3. Рыночная цена (около медианы +/- 10%)          0.15
4. Чуть дороже рынка (на 10-50% дороже)           0.13
5. Намного дороже рынка (в 1.5 - 2 раза дороже)   0.14
6. Оверпрайс (более чем в 2 раза дороже)          0.12
Name: deal_probability, dtype: float64

In [246]:
liq_us_type=df.groupby(['user_type', 'category_name'])['deal_probability'].mean().unstack()
liq_us_type

category_name,Автомобили,Аквариум,Аудио и видео,Билеты и путешествия,Бытовая техника,Велосипеды,Водный транспорт,Гаражи и машиноместа,Готовый бизнес,Грузовики и спецтехника,Детская одежда и обувь,"Дома, дачи, коттеджи",Другие животные,Земельные участки,"Игры, приставки и программы",Квартиры,Книги и журналы,Коллекционирование,Коммерческая недвижимость,Комнаты,Кошки,Красота и здоровье,Мебель и интерьер,Мотоциклы и мототехника,Музыкальные инструменты,Настольные компьютеры,Недвижимость за рубежом,Ноутбуки,Оборудование для бизнеса,"Одежда, обувь, аксессуары",Оргтехника и расходники,Охота и рыбалка,Планшеты и электронные книги,Посуда и товары для кухни,Предложение услуг,Продукты питания,Птицы,Растения,Ремонт и строительство,Собаки,Спорт и отдых,Телефоны,Товары для детей и игрушки,Товары для животных,Товары для компьютера,Фототехника,Часы и украшения
user_type,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Company,0.30,0.11,0.16,0.06,0.18,0.14,0.06,0.11,0.11,0.15,0.06,0.12,0.24,0.10,0.15,0.14,0.02,0.04,0.10,0.14,0.23,0.08,0.13,0.27,0.14,0.16,0.00,0.20,0.09,0.05,0.17,0.08,0.12,0.10,0.42,0.11,0.22,0.14,0.10,0.25,0.11,0.18,0.15,0.11,0.14,0.06,0.03
Private,0.29,0.17,0.18,0.28,0.30,0.31,0.24,0.15,0.10,0.21,0.06,0.13,0.26,0.09,0.23,0.23,0.04,0.07,0.14,0.21,0.30,0.10,0.22,0.25,0.20,0.20,0.24,0.25,0.16,0.05,0.19,0.17,0.22,0.10,0.40,0.15,0.25,0.12,0.17,0.25,0.17,0.20,0.22,0.19,0.18,0.14,0.07
Shop,0.14,0.00,0.03,0.00,0.06,0.00,NaN,0.05,0.02,0.09,0.07,0.07,0.00,0.05,0.03,0.07,0.00,0.00,0.04,0.06,NaN,0.03,0.04,0.09,0.18,0.19,NaN,0.06,0.02,0.03,0.14,0.03,0.06,0.00,0.33,0.11,0.00,0.03,0.06,0.00,0.01,0.08,0.02,0.00,0.08,0.03,0.01


In [247]:
liq_img=df.groupby(['image', 'category_name'])['deal_probability'].mean().unstack()
liq_img

category_name,Автомобили,Аквариум,Аудио и видео,Билеты и путешествия,Бытовая техника,Велосипеды,Водный транспорт,Гаражи и машиноместа,Готовый бизнес,Грузовики и спецтехника,Детская одежда и обувь,"Дома, дачи, коттеджи",Другие животные,Земельные участки,"Игры, приставки и программы",Квартиры,Книги и журналы,Коллекционирование,Коммерческая недвижимость,Комнаты,Кошки,Красота и здоровье,Мебель и интерьер,Мотоциклы и мототехника,Музыкальные инструменты,Настольные компьютеры,Недвижимость за рубежом,Ноутбуки,Оборудование для бизнеса,"Одежда, обувь, аксессуары",Оргтехника и расходники,Охота и рыбалка,Планшеты и электронные книги,Посуда и товары для кухни,Предложение услуг,Продукты питания,Птицы,Растения,Ремонт и строительство,Собаки,Спорт и отдых,Телефоны,Товары для детей и игрушки,Товары для животных,Товары для компьютера,Фототехника,Часы и украшения
image,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,0.19,0.05,0.11,0.28,0.23,0.02,0.30,0.14,0.14,0.12,0.01,0.17,0.22,0.09,0.15,0.21,0.05,0.12,0.15,0.19,0.19,0.15,0.13,0.40,0.01,0.08,0.24,0.34,0.12,0.03,0.10,0.25,0.21,0.09,0.35,0.16,0.21,0.13,0.11,0.22,0.14,0.16,0.17,0.17,0.15,0.00,0.07
1,0.29,0.15,0.17,0.10,0.25,0.24,0.20,0.15,0.08,0.19,0.06,0.10,0.27,0.09,0.20,0.14,0.03,0.05,0.08,0.16,0.29,0.09,0.19,0.25,0.19,0.19,0.00,0.19,0.11,0.05,0.19,0.13,0.17,0.10,0.43,0.13,0.25,0.12,0.14,0.25,0.15,0.18,0.20,0.16,0.16,0.11,0.05


In [248]:
low_liq_cat=df.groupby('category_name')['deal_probability'].mean().reset_index()
low_liq_cat=low_liq_cat.query("deal_probability<0.1")
low_liq_cat

,category_name,deal_probability
8,Готовый бизнес,0.09
10,Детская одежда и обувь,0.06
13,Земельные участки,0.09
16,Книги и журналы,0.04
17,Коллекционирование,0.06
21,Красота и здоровье,0.09
29,"Одежда, обувь, аксессуары",0.05
46,Часы и украшения,0.05
